# Schema Evolution
1. Adding New Columns (Manual / Automatic)
2. Widening Data Types (Supported Delta >= 3.2): Sometimes we need to expand a column's data type to accommodate larger values. Delta Lake allows "widening" type conversions that won't lose data, such as:
- `INT` to `BIGINT`
- `FLOAT` to `DOUBLE`
- `VARCHAR(10)` to `VARCHAR(20)`

3. Nested Structure Evolution (Manual / Automatic): Delta Lake supports evolution of complex data types like structs and arrays. We can:
- Add new fields to structs
- Modify nested field types
- Add new elements to arrays

4. Column Position Changes (Manual / Automatic): we can reorganize our columns

Note: 
- `INSERT` works by matching columns by position
- `MERGE` works by matching columns by name

#### Scenario 1: Adding New Columns 

In [0]:
DROP TABLE deltacatalog.deltadb.invoices_se;

In [0]:
SELECT *
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 1 AND 5

In [0]:
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_se (
  customer_id INT NOT NULL,  
  invoice_no STRING,
  price FLOAT, 
  invoice_date DATE
); 

INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price, invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 1 AND 5

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ADD COLUMNS (quantity INT);

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price, invoice_date, quantity
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 6 AND 10 

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = false;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price, invoice_date, quantity, payment_method 
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 11 AND 15 

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

#### Scenario 2: Type Widening

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se 
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

In [0]:
DESCRIBE TABLE deltacatalog.deltadb.invoices_se; 

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (123456789012345,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card')

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ALTER COLUMN customer_id TYPE BIGINT;

In [0]:
DESCRIBE TABLE deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (123456789012345,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card')

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se
WHERE customer_id = 123456789012345;

#### Scenario 3: Nested Structure Evolution

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ADD COLUMNS purchase_details STRUCT<
  mall_pin_code INT,
  store_code INT
>;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (16,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(12345, 879));

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ALTER COLUMN purchase_details.mall_pin_code TYPE BIGINT;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (17,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(123456789012346, 765));

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
ALTER TABLE deltacatalog.deltadb.invoices_se
ADD COLUMN purchase_details.store_loc STRING;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (17,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(7612, 765, 'ground floor'));

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se 
VALUES (21,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', 
  NAMED_STRUCT(
    'mall_pin_code', 7612, 
    'store_code', 765, 
    'store_loc', 'ground floor', 
    'staff_id', 'ST12736'
  )
);

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

#### Scenario 4: Column Position Changes

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled=false;

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
-- ALTER TABLE deltacatalog.deltadb.invoices_se ADD COLUMNS (age INT FIRST);
ALTER TABLE deltacatalog.deltadb.invoices_se ADD COLUMNS (age INT AFTER price)

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, NULL AS purchase_details
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 50 AND 55

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled = true;

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_se
SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, category, NULL AS purchase_details
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
WHERE customer_id BETWEEN 56 AND 60

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_se tgt
USING (
  SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, category, NULL AS purchase_details
  FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`
  WHERE customer_id BETWEEN 56 AND 60
) src 
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_se;

In [0]:
%python
from pyspark.sql.functions import *
df = (
  spark.read.parquet("abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet")
  .filter(col("customer_id").between(1, 10))
  .select("customer_id", "price", "invoice_date")
)
df.write.saveAsTable("deltacatalog.deltadb.invoices_se_spark_df")

In [0]:
%python
df = (
  spark.read.parquet("abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet")
  .filter(col("customer_id").between(11, 25))
  .select("customer_id", "price", "invoice_date", "quantity", "payment_method")
)
df.write.mode("append").option("mergeSchema", "true").saveAsTable("deltacatalog.deltadb.invoices_se_spark_df")